# Create Sumitomo Foundation Awards (公益財団法人住友財団)

Creates Sumitomo Foundation (Tokyo) research grants from the foundation's own
per-year grantee lists on https://www.sumitomo.or.jp/ (static Shift-JIS HTML
tables; no export or API). **2,700 grants, FY2010-FY2025, 100% title / amount /
PI / fiscal year, 99.9% institution** (local run 2026-10-01, oxjob #1451).

| programme | FY | grants | amount unit on the page |
|---|---|---|---|
| 基礎科学研究助成 Basic Science Research | 2014-2025 | 1,106 | 万円 (x10,000 JPY) |
| 環境研究助成 Environmental Research (課題研究 + 一般研究) | 2014-2025 | 501 | 万円 |
| アジア諸国における日本関連研究助成 Japan-Related Research in Asian Countries | 2010-2024 | 1,093 | 千円 (x1,000 JPY; FY2024 円) or USD |

Every parsed per-page count matches the page's own 合計 total (jare FY2010 lists
81 rows under an "82 件" total: list number 48 is absent from the page).

**Filter (stated per batch rules):** the three cultural-property programmes
(文化財維持・修復事業助成, 海外の文化財維持・修復事業助成, 修復文化財展示事業助成) are
**excluded**: their lists are conservation/repair jobs on named objects (e.g. a
Kamakura-period saddle at Kashima Shrine, a screen at a US museum) paid to temples,
municipalities and museums, with no researcher or research question. Jare FY2025 is
published only as a PDF whose amount column is not row-aligned and is not parsed
(known gap, ~30 grants). Years before FY2014 (kiso/kankyo) and FY2010 (jare) are
only in the annual-report PDFs; not ingested.

**Prerequisites:** run `scripts/local/sumitomo_foundation_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/sumitomo_foundation/sumitomo_foundation_projects.parquet`
(§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** citing works quote the Sumitomo grant number
(`180853`, `214019`; 7 digits from FY2022, `2200735`): 408 of the 585 distinct
acknowledgement stubs under F4320310657 are that shape. The site publishes the
number only in the file names of the results-report PDFs linked from the FY2014-FY2022
lists, so 1,306 grants (kiso 884, kankyo 382, jare 40) ship the published number
and **264 existing stubs collapse onto this ingest**. Grants without a published
number (FY2023-FY2025 lists, all jare before FY2019, most jare after, kankyo 課題研究)
ship a stable synthetic key `SF-{KISO|KANKYO|JARE}-{FY}-{md5(name|title)[:8]}`.
Three report numbers linked from more than one row are kept only where the PDF's list
position matches the row (kiso FY2017 170502), else dropped to the synthetic key
(kiso FY2015 150743, jare FY2019 198401).

**Fields:** amount converted to whole JPY in the script (万円 x10,000, 千円 x1,000);
USD rows (jare) keep USD. `start_year` = Sumitomo fiscal year of the award (April-March;
grants are decided at the autumn board meeting); the lists carry no start/end dates, so
`start_date`/`end_date` are NULL rather than invented. Names are printed family-first
in the lists (浅香 猛 -> family 浅香); dot-joined katakana transliterations keep the
source's given-first order. `affiliation.country` = JP for kiso/kankyo (domestic
programmes); jare uses the printed country/region (ISO-2), preferring the institution's
own ＜country＞ when given. `funding_type` = research.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320310657`
- display_name: Sumitomo Foundation
- ror_id / doi: from `openalex.funders.funders` (ror 01nmpj461, doi 10.13039/100008608)

**Priority:** `540` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.sumitomo_foundation_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/sumitomo_foundation/sumitomo_foundation_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_rows FROM openalex.awards.sumitomo_foundation_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.sumitomo_foundation_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.sumitomo_foundation_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320310657 is a Crossref-registered F4320* funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320310657;

## Step 2: Create Sumitomo Foundation Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.sumitomo_foundation_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320310657  -- Sumitomo Foundation
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN g.currency END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    g.funder_scheme as funder_scheme,
    'sumitomo_foundation_grant_lists' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.fiscal_year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    CASE WHEN NULLIF(TRIM(g.lead_family_name), '') IS NOT NULL THEN
        named_struct(
            'given_name', NULLIF(TRIM(g.lead_given_name), ''),
            'family_name', NULLIF(TRIM(g.lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(g.lead_institution), ''),
                'country', NULLIF(TRIM(g.lead_country_code), ''),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        )
    END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN NULLIF(TRIM(g.lead_family_name), '') IS NOT NULL THEN array(
        named_struct(
            'given_name', NULLIF(TRIM(g.lead_given_name), ''),
            'family_name', NULLIF(TRIM(g.lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(g.lead_institution), ''),
                'country', NULLIF(TRIM(g.lead_country_code), ''),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        )
    ) END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.sumitomo_foundation_raw g
CROSS JOIN src_funder f
WHERE NULLIF(TRIM(g.funder_award_id), '') IS NOT NULL;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'sumitomo_foundation_grant_lists' AND priority = 540;

-- Insert into openalex_awards_raw with priority.
-- Priority 540: direct-from-funder ingest (higher wins under the 2026-06-20 DESC
-- dedup, oxjob #500), so it outranks the priority-0 acknowledgement shells that
-- share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    540 as priority
FROM openalex.awards.sumitomo_foundation_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids,
       COUNT(DISTINCT LOWER(funder_award_id)) as distinct_award_ids
FROM openalex.awards.sumitomo_foundation_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, lead_investigator.affiliation.country
FROM openalex.awards.sumitomo_foundation_awards LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage
SELECT
    COUNT(*) as total,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) as pct_title,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    collect_set(currency) as currencies,
    MIN(amount) as min_amt, MAX(amount) as max_amt, ROUND(AVG(amount), 0) as avg_amt,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) as pct_start_date,
    ROUND(COUNT(start_year) * 100.0 / COUNT(*), 1) as pct_start_year,
    ROUND(COUNT(lead_investigator.family_name) * 100.0 / COUNT(*), 1) as pct_pi,
    ROUND(COUNT(lead_investigator.affiliation.name) * 100.0 / COUNT(*), 1) as pct_institution
FROM openalex.awards.sumitomo_foundation_awards;

In [ ]:
%sql
SELECT funder_scheme, currency, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_amount
FROM openalex.awards.sumitomo_foundation_awards
GROUP BY funder_scheme, currency ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.sumitomo_foundation_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.sumitomo_foundation_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.sumitomo_foundation_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.5 funder consistency
SELECT funder.display_name, funder_id, COUNT(*)
FROM openalex.awards.sumitomo_foundation_awards
GROUP BY funder.display_name, funder_id ORDER BY 3 DESC;

In [ ]:
%sql
-- Award-id shape (2.1.1): published Sumitomo numbers are 6 digits (to FY2021) or
-- 7 digits (FY2022+); everything else must be the documented synthetic key.
SELECT
    SUM(CASE WHEN funder_award_id RLIKE '^[0-9]{6,7}$' THEN 1 ELSE 0 END) AS published_number,
    SUM(CASE WHEN funder_award_id RLIKE '^SF-(KISO|KANKYO|JARE)-[0-9]{4}-[0-9a-f]{8}$' THEN 1 ELSE 0 END) AS synthetic_key,
    SUM(CASE WHEN NOT (funder_award_id RLIKE '^[0-9]{6,7}$' OR funder_award_id RLIKE '^SF-(KISO|KANKYO|JARE)-[0-9]{4}-[0-9a-f]{8}$') THEN 1 ELSE 0 END) AS other_shape -- must be 0
FROM openalex.awards.sumitomo_foundation_awards;

In [ ]:
%sql
-- Existing acknowledgement stubs that collapse onto this ingest (expect ~264).
SELECT c.provenance, COUNT(*) AS stub_rows, COUNT(s.id) AS collapsed
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.sumitomo_foundation_awards s ON s.id = c.id
WHERE c.funder_id = 4320310657 AND c.priority = 0
GROUP BY c.provenance;

In [ ]:
%sql
-- Section 6.8: rows reached the shared raw table at the assigned priority.
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'sumitomo_foundation_grant_lists'
GROUP BY provenance, priority;